# Tema 6 - Laboratorul 7
## Exercitiul 1. Controlul calitatii intr-o fabrica

### a) Modelul probabilistic

Fiecare componenta testata este defecta cu probabilitatea $\theta$, independent de celelalte. Deci numarul de componente defecte din $n = 50$ este o variabila binomiala:

$$Y \mid \theta \sim Bin(50, \theta)$$

Functia de verosimilitate:

$$P(Y = y \mid \theta) = \binom{50}{y} \theta^{y} (1-\theta)^{50-y}$$

Pentru $y = 8$:

$$P(Y = 8 \mid \theta) = \binom{50}{8} \theta^{8} (1-\theta)^{42}$$

### b) Actualizarea Bayesiana

Formula lui Bayes: $p(\theta \mid y) \propto P(Y = y \mid \theta) \cdot p(\theta)$.

Distributia a priori este $Beta(2, 18)$, deci

$$p(\theta) = \frac{\theta^{2-1} (1-\theta)^{18-1}}{B(2, 18)} = \frac{\theta (1-\theta)^{17}}{B(2, 18)}$$

Inmultim cu verosimilitatea si pastram doar factorii care depind de $\theta$ (ceilalti sunt constante):

$$p(\theta \mid y = 8) \propto \theta^{8} (1-\theta)^{42} \cdot \theta^{1} (1-\theta)^{17} = \theta^{9} (1-\theta)^{59}$$

Aceasta este forma unei densitati $Beta(a, b)$ cu $a - 1 = 9$ si $b - 1 = 59$, adica $a = 10$ si $b = 60$. Constanta de normalizare se obtine automat, deoarece densitatea trebuie sa integreze la 1.

In general, daca $\theta \sim Beta(\alpha, \beta)$ si $Y \mid \theta \sim Bin(n, \theta)$, atunci

$$\theta \mid Y = y \sim Beta(\alpha + y, \beta + n - y)$$

Aici: $\alpha + y = 2 + 8 = 10$ si $\beta + n - y = 18 + 50 - 8 = 60$.

$$\boxed{\theta \mid Y = 8 \sim Beta(10, 60)}$$

Prin conjugare, a priori si a posteriori sunt din aceeasi familie (Beta), de aceea calculul a fost atat de simplu.

### c) Estimarea parametrului

**MLE.** Maximizam verosimilitatea. Logaritmul ei (fara constanta) este $y \ln\theta + (n-y)\ln(1-\theta)$. Derivam in raport cu $\theta$ si egalam cu 0:

$$\frac{y}{\theta} - \frac{n-y}{1-\theta} = 0 \;\Rightarrow\; y(1-\theta) = (n-y)\theta \;\Rightarrow\; y = n\theta$$

$$\hat\theta_{MLE} = \frac{y}{n} = \frac{8}{50} = 0.16$$

**MAP.** Este modul distributiei a posteriori $Beta(10, 60)$. Pentru $Beta(a, b)$ cu $a, b > 1$, modul este $\frac{a-1}{a+b-2}$:

$$\hat\theta_{MAP} = \frac{10 - 1}{10 + 60 - 2} = \frac{9}{68} \approx 0.1324$$

**Media a posteriori.** Media unei $Beta(a, b)$ este $\frac{a}{a+b}$:

$$E[\theta \mid Y = 8] = \frac{10}{10 + 60} = \frac{10}{70} \approx 0.1429$$

**Comparatie.**

| estimator | valoare |
|---|---|
| MLE | 0.1600 |
| MAP | 0.1324 |
| media a posteriori | 0.1429 |

MLE foloseste doar datele (8 defecte din 50). MAP si media a posteriori combina datele cu informatia a priori, a carei medie este $\frac{2}{20} = 0.1$, mai mica decat 0.16. De aceea ambele sunt trase spre 0.1.

Media a posteriori se poate scrie ca o medie ponderata intre media a priori si MLE:

$$\frac{10}{70} = \frac{20}{70} \cdot 0.1 + \frac{50}{70} \cdot 0.16 = 0.0286 + 0.1143 = 0.1429$$

Ponderile sunt $\frac{20}{70}$ pentru prior (echivalent cu 20 de observatii) si $\frac{50}{70}$ pentru date (50 de observatii). MAP este putin mai mic decat media pentru ca distributia $Beta(10, 60)$ este asimetrica la dreapta.

### d), e), f) Cod: grafic, interval credibil, probabilitate a posteriori

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

alfa_apriori = 2
beta_apriori = 18
nr_testate = 50
nr_defecte = 8

alfa_aposteriori = alfa_apriori + nr_defecte
beta_aposteriori = beta_apriori + nr_testate - nr_defecte

mle = nr_defecte / nr_testate
map_estimat = (alfa_aposteriori - 1) / (alfa_aposteriori + beta_aposteriori - 2)
media_aposteriori = alfa_aposteriori / (alfa_aposteriori + beta_aposteriori)
media_apriori = alfa_apriori / (alfa_apriori + beta_apriori)

print("MLE:", mle)
print("MAP:", map_estimat)
print("Media a posteriori:", media_aposteriori)
print("Media a priori:", media_apriori)

valori_theta = np.linspace(0, 0.5, 500)
densitate_apriori = beta.pdf(valori_theta, alfa_apriori, beta_apriori)
densitate_aposteriori = beta.pdf(valori_theta, alfa_aposteriori, beta_aposteriori)

plt.plot(valori_theta, densitate_apriori, label="a priori Beta(2, 18)")
plt.plot(valori_theta, densitate_aposteriori, label="a posteriori Beta(10, 60)")
plt.axvline(mle, color="gray", linestyle="--", label="MLE")
plt.xlabel("theta")
plt.ylabel("densitate")
plt.legend()
plt.show()

limita_stanga = beta.ppf(0.025, alfa_aposteriori, beta_aposteriori)
limita_dreapta = beta.ppf(0.975, alfa_aposteriori, beta_aposteriori)
print("Interval credibil 95%:", limita_stanga, limita_dreapta)

prob_sub_10 = beta.cdf(0.10, alfa_aposteriori, beta_aposteriori)
print("P(theta < 0.10 | Y = 8):", prob_sub_10)

prob_predictiva = 1.0
for i in range(10):
    prob_predictiva = prob_predictiva * (60 + i) / (70 + i)
print("P(X = 0 | Y = 8):", prob_predictiva)

**d) Interpretare.** Densitatea a priori $Beta(2, 18)$ are media 0.1 si este destul de larga. Dupa observarea a 8 componente defecte din 50 (rata observata 16%, mai mare decat 10%), densitatea a posteriori $Beta(10, 60)$ s-a mutat spre dreapta, in jurul valorii 0.14, si a devenit mai ingusta. Datele au crescut deci estimarea ratei de defectare si au redus incertitudinea.

**e) Interval credibil.** Din rulare, intervalul central de 95% este aproximativ $(0.0717,\ 0.2332)$. Interpretare: dupa ce am vazut datele, exista probabilitate 95% ca rata reala de defectare sa fie intre aproximativ 7.2% si 23.3%. Intervalul este destul de larg, pentru ca avem doar 50 de componente testate.

**f) Decizie practica.** Din rulare, $P(\theta < 0.10 \mid Y = 8) \approx 0.148$. Deci doar aproximativ 14.8% sansa ca rata de defectare sa fie sub 10%. Datele nu sustin cerinta fabricii: este mult mai probabil ca rata sa fie peste 10%. Ar fi nevoie de mai multe teste sau de imbunatatirea procesului de productie.

### g) Predictia Bayesiana

Daca $X$ este numarul de componente defecte din 10 noi, atunci $P(X = 0 \mid \theta) = (1-\theta)^{10}$. Folosim indiciul:

$$P(X = 0 \mid Y = 8) = \int_0^1 (1-\theta)^{10} \, p(\theta \mid Y = 8) \, d\theta = \int_0^1 (1-\theta)^{10} \cdot \frac{\theta^{9} (1-\theta)^{59}}{B(10, 60)} \, d\theta$$

$$= \frac{1}{B(10, 60)} \int_0^1 \theta^{9} (1-\theta)^{69} \, d\theta = \frac{B(10, 70)}{B(10, 60)}$$

Aici am folosit faptul ca integrala unei densitati Beta nenormalizate este functia $B$: $\int_0^1 \theta^{a-1}(1-\theta)^{b-1} d\theta = B(a, b)$, cu $a = 10$ si $b = 70$.

Scriem $B(a, b) = \frac{\Gamma(a)\Gamma(b)}{\Gamma(a+b)}$:

$$\frac{B(10, 70)}{B(10, 60)} = \frac{\Gamma(10)\Gamma(70)}{\Gamma(80)} \cdot \frac{\Gamma(70)}{\Gamma(10)\Gamma(60)} = \frac{\Gamma(70)}{\Gamma(60)} \cdot \frac{\Gamma(70)}{\Gamma(80)}$$

Folosim $\Gamma(n+1) = n\,\Gamma(n)$:

$$\frac{\Gamma(70)}{\Gamma(60)} = 60 \cdot 61 \cdots 69, \qquad \frac{\Gamma(80)}{\Gamma(70)} = 70 \cdot 71 \cdots 79$$

$$P(X = 0 \mid Y = 8) = \frac{60 \cdot 61 \cdot 62 \cdot 63 \cdot 64 \cdot 65 \cdot 66 \cdot 67 \cdot 68 \cdot 69}{70 \cdot 71 \cdot 72 \cdot 73 \cdot 74 \cdot 75 \cdot 76 \cdot 77 \cdot 78 \cdot 79}$$

Grupand fractiile:

$$= \frac{60}{70} \cdot \frac{61}{71} \cdot \frac{62}{72} \cdot \frac{63}{73} \cdot \frac{64}{74} \cdot \frac{65}{75} \cdot \frac{66}{76} \cdot \frac{67}{77} \cdot \frac{68}{78} \cdot \frac{69}{79}$$

Valorile fractiilor sunt aproximativ 0.8571, 0.8592, 0.8611, 0.8630, 0.8649, 0.8667, 0.8684, 0.8701, 0.8718, 0.8734. Produsul lor:

$$P(X = 0 \mid Y = 8) \approx 0.236$$

(valoare verificata si in cod, in celula de mai sus). Deci exista aproximativ 23.6% sansa ca din 10 componente noi niciuna sa nu fie defecta.

## Exercitiul 2. Restaurant fast-food (simulare Monte Carlo)

### a) Modelul simularii

- Sosirile formeaza un proces Poisson cu $\lambda = 20/60 = 1/3$ clienti pe minut. Numarul de clienti dintr-o ora este Poisson cu media 20, iar momentele sosirilor, dat fiind numarul lor, sunt uniforme pe $[0, 60]$ (sortate).
- Timpul la casa este exponential cu media 2 minute, iar timpul de preparare este exponential cu media $\alpha$ minute.

**Cele doua cozi (FIFO).**

1. *Casa:* clientul $i$ incepe sa plateasca la momentul `max(sosire_i, sfarsit_plata_(i-1))`, adica fie cand ajunge (daca casa e libera), fie cand termina clientul dinaintea lui. Apoi `sfarsit_plata_i = start_plata_i + timp_plata_i`.
2. *Statia de gatit:* comanda clientului $i$ intra in coada la `sfarsit_plata_i`. Gatitul incepe la `max(sfarsit_plata_i, sfarsit_gatit_(i-1))` si se termina dupa timpul de preparare. Momentul `sfarsit_gatit_i` este momentul la care primeste mancarea.

Deoarece casa este FIFO, comenzile ajung la gatit in aceeasi ordine in care au sosit clientii, deci si coada de la gatit este FIFO in aceeasi ordine.

Pentru ca sa se poata compara usor mai multe valori ale lui $\alpha$, generez o singura data toate simularile, cu timpii de preparare "unitari" (exponentiala cu media 1), si ii inmultesc cu $\alpha$ de fiecare data. Un timp exponential cu media $\alpha$ este $\alpha$ ori un timp exponential cu media 1.

In [ ]:
import numpy as np

generator = np.random.default_rng(2026)

rata_sosiri = 20 / 60
durata_ora = 60
medie_plata = 2
limita_timp = 15
prag_probabilitate = 0.95
nr_simulari = 5000


def genereaza_ora():
    nr_clienti = generator.poisson(rata_sosiri * durata_ora)
    momente_sosire = np.sort(generator.uniform(0, durata_ora, nr_clienti))
    timpi_plata = generator.exponential(medie_plata, nr_clienti)
    timpi_preparare_unitari = generator.exponential(1, nr_clienti)
    return momente_sosire, timpi_plata, timpi_preparare_unitari


def calculeaza_servirea(momente_sosire, timpi_plata, timpi_preparare):
    nr_clienti = len(momente_sosire)
    timp_total = np.zeros(nr_clienti)
    timp_asteptare = np.zeros(nr_clienti)
    sfarsit_casa_anterior = 0
    sfarsit_gatit_anterior = 0
    for i in range(nr_clienti):
        start_plata = max(momente_sosire[i], sfarsit_casa_anterior)
        sfarsit_plata = start_plata + timpi_plata[i]
        start_gatit = max(sfarsit_plata, sfarsit_gatit_anterior)
        sfarsit_gatit = start_gatit + timpi_preparare[i]
        timp_total[i] = sfarsit_gatit - momente_sosire[i]
        timp_asteptare[i] = (start_plata - momente_sosire[i]) + (start_gatit - sfarsit_plata)
        sfarsit_casa_anterior = sfarsit_plata
        sfarsit_gatit_anterior = sfarsit_gatit
    return timp_total, timp_asteptare


simulari = []
for k in range(nr_simulari):
    simulari.append(genereaza_ora())


def probabilitate_toti_ok(alfa):
    nr_ore_bune = 0
    for momente_sosire, timpi_plata, timpi_preparare_unitari in simulari:
        timpi_preparare = alfa * timpi_preparare_unitari
        timp_total, timp_asteptare = calculeaza_servirea(momente_sosire, timpi_plata, timpi_preparare)
        if len(timp_total) == 0 or timp_total.max() <= limita_timp:
            nr_ore_bune = nr_ore_bune + 1
    return nr_ore_bune / nr_simulari


for alfa_test in [0.5, 1.0, 1.5, 2.0, 2.5]:
    print("alfa =", alfa_test, "probabilitate =", probabilitate_toti_ok(alfa_test))


def probabilitate_client(alfa):
    nr_clienti_ok = 0
    nr_clienti_total = 0
    for momente_sosire, timpi_plata, timpi_preparare_unitari in simulari:
        timp_total, timp_asteptare = calculeaza_servirea(momente_sosire, timpi_plata, alfa * timpi_preparare_unitari)
        nr_clienti_ok = nr_clienti_ok + np.sum(timp_total <= limita_timp)
        nr_clienti_total = nr_clienti_total + len(timp_total)
    return nr_clienti_ok / nr_clienti_total


print("probabilitate toti clientii, alfa foarte mic:", probabilitate_toti_ok(0.001))

alfa_mic = 0.001
alfa_mare = 3.0
for pas in range(12):
    alfa_mijloc = (alfa_mic + alfa_mare) / 2
    if probabilitate_client(alfa_mijloc) >= prag_probabilitate:
        alfa_mic = alfa_mijloc
    else:
        alfa_mare = alfa_mijloc

alfa_final = alfa_mic
print("alfa final (client ales la intamplare):", alfa_final)
print("probabilitate client:", probabilitate_client(alfa_final))
print("probabilitate toti clientii:", probabilitate_toti_ok(alfa_final))

toti_timpii = []
toate_asteptarile = []
for momente_sosire, timpi_plata, timpi_preparare_unitari in simulari:
    timp_total, timp_asteptare = calculeaza_servirea(momente_sosire, timpi_plata, alfa_final * timpi_preparare_unitari)
    toti_timpii.extend(timp_total)
    toate_asteptarile.extend(timp_asteptare)

print("timp mediu pana la primirea mancarii:", np.mean(toti_timpii))
print("timp mediu de asteptare:", np.mean(toate_asteptarile))

### b) Rezultate si observatie importanta

Din rulare, probabilitatea ca **toti** clientii dintr-o ora sa primeasca mancarea in maximum 15 minute este:

| $\alpha$ | probabilitate |
|---|---|
| 0.5 | 0.787 |
| 1.0 | 0.731 |
| 1.5 | 0.621 |
| 2.0 | 0.439 |
| 2.5 | 0.277 |

Probabilitatea scade cand $\alpha$ creste, cum ne asteptam. Dar chiar si pentru $\alpha$ foarte aproape de 0 (statia de gatit practic instantanee) probabilitatea este de aproximativ 0.81, deci **sub 95%**. Motivul este ca doar casa (cu timp mediu 2 minute si sosiri la 3 minute, deci incarcare $\frac{2}{3}$) produce uneori asteptari lungi: cand mai multi clienti sosesc aproape simultan, ultimii ajung sa astepte peste 15 minute doar la casa.

Concluzie: **pentru parametrii din enunt, nu exista nicio valoare $\alpha$ pentru care conditia de 95% pentru toti clientii sa fie indeplinita.** Cea mai mare probabilitate posibila este aproximativ 0.81.

Daca in schimb cerem ca un client *ales la intamplare* sa primeasca mancarea in 15 minute cu probabilitate cel putin 95%, cautarea prin bisectie (intervalul $[0.001, 3]$, 12 pasi) da

$$\alpha \approx 0.29 \text{ minute}$$

pentru care probabilitatea per client este 0.950, iar probabilitatea pentru toti clientii dintr-o ora este tot doar aproximativ 0.80.

### c) Timpi medii (pentru $\alpha \approx 0.29$)

- timpul mediu intre sosire si primirea mancarii: aproximativ 5.26 minute
- timpul mediu de asteptare efectiva (fara servire): aproximativ 2.98 minute

### d) Detalii despre simulare

- numar de simulari: 5000 de ore
- samanta aleatoare: 2026 (`np.random.default_rng(2026)`)
- cautarea lui $\alpha$: bisectie pe intervalul $[0.001, 3]$, 12 iteratii, folosind aceleasi simulari (cu timpi unitari) la fiecare pas
- Estimarile se schimba de la o rulare la alta pentru ca simularea foloseste numere aleatoare: alta samanta sau alt numar de simulari inseamna alte esantioane, deci alte frecvente relative. Eroarea scade aproximativ cu $\frac{1}{\sqrt{N}}$ cand cresc numarul de simulari $N$. Pentru probabilitati apropiate de 0.95 diferentele pot muta vizibil valoarea lui $\alpha$ gasita prin bisectie.